In [ ]:
# Experiment builder: submits JK/lf_tweezer_evap_t_check.py once per post_abs_wait value.
# Modeled on the ExptBuilder in "Mloop testing/integration/lf_exp.py", but instead of
# templating a copy of the experiment, it reads the source file and swaps out the
# `self.p.post_abs_wait = ...` line, so edits to the experiment carry over.
import os
import re
import time
from subprocess import PIPE, run

import numpy as np

In [ ]:
post_abs_wait_values = [0., 0.1, 1., 2., 3., 4., 5., 6., 7., 10.]  # seconds
t_cooldown_after_first = 60.  # seconds, wait after the first experiment ends

In [ ]:
class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__expt_dir__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments")
        self.__source_path__ = os.path.join(self.__expt_dir__, "JK", "lf_tweezer_evap_t_check.py")
        self.__temp_exp_path__ = os.path.join(self.__expt_dir__, "post_abs_wait_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run --device-db %db% " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode

    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)

    def post_abs_wait_expt(self, post_abs_wait):
        """Returns the source of lf_tweezer_evap_t_check.py with
        self.p.post_abs_wait set to the given value (s)."""
        with open(self.__source_path__, 'r') as file:
            source = file.read()
        pattern = r"^(\s*)self\.p\.post_abs_wait\s*=.*$"
        script, n = re.subn(pattern, rf"\g<1>self.p.post_abs_wait = {float(post_abs_wait)}",
                            source, flags=re.MULTILINE)
        if n != 1:
            raise ValueError(f"Expected exactly one 'self.p.post_abs_wait = ...' line in {self.__source_path__}, found {n}.")
        return script

eBuilder = ExptBuilder()

In [ ]:
# preview the substituted line before submitting
print([line for line in eBuilder.post_abs_wait_expt(post_abs_wait_values[0]).splitlines()
       if 'post_abs_wait' in line])

In [ ]:
returncodes = []
for i, post_abs_wait in enumerate(post_abs_wait_values):
    print(f"--- run {i+1}/{len(post_abs_wait_values)}: post_abs_wait = {post_abs_wait} s ---")
    eBuilder.write_experiment_to_file(eBuilder.post_abs_wait_expt(post_abs_wait))
    returncode = eBuilder.run_expt()
    returncodes.append(returncode)
    if returncode != 0:
        print(f"Experiment failed (return code {returncode}), stopping.")
        break
    if i == 0:
        print(f"Waiting {t_cooldown_after_first:.0f} s for the machine to cool down...")
        time.sleep(t_cooldown_after_first)

print(dict(zip(post_abs_wait_values, returncodes)))